# 医疗场景模型安全性对齐训练 - 结果分析

本 Notebook 用于分析 Base / SFT / DPO 三个模型的对比结果。

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. 加载数据

加载评估报告和原始评判数据。

In [ ]:
# 配置路径
RESULTS_DIR = Path('../results')
REPORT_PATH = RESULTS_DIR / 'reports/evaluation_report.json'

# 加载报告
with open(REPORT_PATH, 'r', encoding='utf-8') as f:
    report = json.load(f)

report.keys()

## 2. 总体分数对比

In [ ]:
# 创建 DataFrame
scores_df = pd.DataFrame(report['dimension_scores']).T
scores_df = scores_df[['safety', 'accuracy', 'helpfulness', 'harmlessness', 'medical_correctness', 'overall']]

scores_df

In [ ]:
# 柱状图
ax = scores_df.plot(kind='bar', width=0.8)
ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison')
ax.set_ylim(0, 10)
plt.xticks(rotation=0)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 3. 安全违规率分析

In [ ]:
violations = pd.Series(report['safety_violations']) * 100

fig, ax = plt.subplots(figsize=(8, 6))
colors = ['green' if v < 10 else 'orange' if v < 30 else 'red' for v in violations]
violations.plot(kind='bar', ax=ax, color=colors)
ax.set_ylabel('Violation Rate (%)')
ax.set_title('Safety Violation Rate by Model')
ax.axhline(y=10, color='g', linestyle='--', alpha=0.5, label='Good (<10%)')
ax.axhline(y=30, color='r', linestyle='--', alpha=0.5, label='Poor (>30%)')
ax.legend()
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 4. 按类别分析

In [ ]:
# 整理类别数据
category_data = []
for model, cats in report['category_analysis'].items():
    for cat, scores in cats.items():
        category_data.append({
            'Model': model,
            'Category': cat,
            'Safety': scores.get('safety', 0)
        })

cat_df = pd.DataFrame(category_data)
cat_pivot = cat_df.pivot(index='Category', columns='Model', values='Safety')
cat_pivot

In [ ]:
# 热力图
plt.figure(figsize=(10, 6))
sns.heatmap(cat_pivot, annot=True, fmt='.2f', cmap='RdYlGn', vmin=0, vmax=10)
plt.title('Safety Score by Category and Model')
plt.tight_layout()
plt.show()

## 5. 案例分析

查看具体问题的回答和评分。

In [ ]:
# 加载原始评判数据
JUDGMENTS_DIR = RESULTS_DIR / 'judgments'

all_judgments = {}
for file in JUDGMENTS_DIR.glob('*_judgments.json'):
    model_name = file.stem.replace('_judgments', '')
    with open(file, 'r') as f:
        all_judgments[model_name] = json.load(f)

# 查看第一个问题的对比
if all_judgments:
    first_q = list(all_judgments.keys())[0]
    sample = all_judgments[first_q][0]
    
    print(f"问题: {sample['question']}")
    print(f"\n类别: {sample['category']}")
    print(f"期望行为: {sample['metadata']['expected_behavior']}")
    
    print("\n" + "="*60)
    print("各模型回答对比:")
    print("="*60)
    
    for model, judgments in all_judgments.items():
        response = judgments[0]['response']
        score = judgments[0]['judgment'].get('overall', 0)
        print(f"\n{model.upper()} (Score: {score}):")
        print(response[:300] + "..." if len(response) > 300 else response)

## 6. 改进分析

对比 Base -> SFT -> DPO 的改进。

In [ ]:
# 假设顺序: base -> sft -> dpo
models_order = ['base', 'sft', 'dpo']

if all(m in scores_df.index for m in models_order):
    
    improvements = pd.DataFrame({
        'SFT vs Base': scores_df.loc['sft'] - scores_df.loc['base'],
        'DPO vs SFT': scores_df.loc['dpo'] - scores_df.loc['sft'],
        'DPO vs Base': scores_df.loc['dpo'] - scores_df.loc['base']
    })
    
    print("各阶段改进:")
    print(improvements.round(2))
    
    # 可视化
    ax = improvements.plot(kind='bar')
    ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    ax.set_ylabel('Score Improvement')
    ax.set_title('Model Improvement Through SFT and DPO')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 7. 结论

根据以上分析，总结：

1. **安全性提升**: DPO 训练是否显著提高了安全性？
2. **准确性保持**: SFT 和 DPO 是否保持了医疗知识的准确性？
3. **有用性平衡**: 安全性的提升是否以牺牲有用性为代价？

请根据实际实验结果填写具体结论。